# 🫀 Anatomy Splat: turn a specimen video into a 3D Gaussian splat

This notebook takes a phone video of a specimen and produces a **3D Gaussian splat** (`.ply`) that you can view, clean up and share in [SuperSplat](https://superspl.at/editor). An optional last section turns the splat into a **mesh** (`.ply` / `.stl`) for 3D printing or other 3D software.

**Before you start**
1. Film your specimen. Follow the filming tips in the [README](https://github.com/Brundage-VAIL/anatomy_splat#1-film-your-specimen).
2. In Google Drive, create the folder **`My Drive/anatomy_splat/<your project name>/videos/`** and upload your video(s) to it.
3. In Colab, go to **Runtime → Change runtime type**, choose **T4 GPU** (or better), and click **Save**.

**How to run:** run the cells in order, top to bottom. To run a cell, click ▶ or press **Shift+Enter**. Wait for each cell to finish (✅) before you start the next one. Expect about **1–1.5 hours** in total. Most of that is training, and you can leave the tab open while it runs.

| Step | What happens | Time (T4) |
|---|---|---|
| ① Settings | name your project | — |
| ② Connect | check GPU, connect Google Drive | 1 min |
| ③ Install | install COLMAP, gsplat, etc. | 5–10 min |
| ④ Frames | pick sharp frames from your video | 1–3 min |
| ⑤ COLMAP | work out where the camera was for each frame | 10–30 min |
| ⑥ Train | optimize the Gaussian splat | 30–60 min |
| ⑦ Export | save the `.ply` to your Drive | 1 min |
| ⑧ Mesh *(optional)* | 2DGS training → surface mesh | 45–75 min |

Everything you need to keep is saved to `My Drive/anatomy_splat/<project>/outputs/`.

In [ ]:
#@title ① Settings: edit these, then run this cell
#@markdown **PROJECT_NAME** must match your folder name in `My Drive/anatomy_splat/`. Use no spaces.
PROJECT_NAME = "my_specimen"  #@param {type:"string"}
#@markdown How many frames to take from the video. 100–200 is usually right.
NUM_FRAMES = 150  #@param {type:"slider", min:50, max:400, step:10}
#@markdown Longest image side in pixels. Higher means more detail but slower training and more GPU memory.
MAX_IMAGE_SIZE = 1600  #@param [1200, 1600, 2000] {type:"raw"}
#@markdown Training length. Use 7000 for a quick test run and 30000 for the final result.
TRAIN_STEPS = 30000  #@param [7000, 15000, 30000] {type:"raw"}
#@markdown Leave this on `exhaustive` unless you use more than 300 frames.
MATCHER = "exhaustive"  #@param ["exhaustive", "sequential"]

from pathlib import Path
import subprocess

DRIVE_ROOT = Path("/content/drive/MyDrive/anatomy_splat")
PROJECT_DIR = DRIVE_ROOT / PROJECT_NAME
VIDEO_DIR = PROJECT_DIR / "videos"       # you upload videos here
OUT_DIR = PROJECT_DIR / "outputs"        # results are saved here
WORK = Path("/content/work") / PROJECT_NAME   # fast scratch space (erased when Colab disconnects)
REPO = Path("/content/anatomy_splat")
GSPLAT = Path("/content/gsplat-repo")
STEPS_SCALER = TRAIN_STEPS / 30000


def run(cmd, cwd=None):
    """Run a shell command, show its output, and stop with a clear error if it fails."""
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Command failed (exit code {p.returncode}). Read the messages above for the cause.")


print(f"Project: {PROJECT_NAME}  |  {NUM_FRAMES} frames @ {MAX_IMAGE_SIZE}px  |  {TRAIN_STEPS} training steps")

In [ ]:
#@title ② Check the GPU and connect Google Drive
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError("No GPU found. Go to Runtime → Change runtime type → T4 GPU, then Runtime → Run all.")
print("GPU:", gpu.stdout.strip())

from google.colab import drive
drive.mount("/content/drive")

VIDEO_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
inputs = [f.name for f in VIDEO_DIR.iterdir() if not f.name.startswith(".")]
if not inputs:
    raise RuntimeError(f"No videos found. Upload your video(s) to Google Drive → My Drive/anatomy_splat/"
                       f"{PROJECT_NAME}/videos/ and run this cell again.")
print(f"Found in {VIDEO_DIR}:", ", ".join(inputs))

In [ ]:
#@title ③ Install software (5–10 min, needed every time Colab restarts)
run(f"git clone -q --depth 1 https://github.com/Brundage-VAIL/anatomy_splat.git {REPO} || git -C {REPO} pull -q")
run(f"bash {REPO}/scripts/colab_setup.sh")

## ④ Pick frames from the video
The notebook splits the video into `NUM_FRAMES` equal time windows and keeps the **sharpest** frame from each one, skipping blurry frames. After it runs, check the preview: you should see the specimen from many different angles.

In [ ]:
#@title ④ Extract frames
run(f'python {REPO}/scripts/extract_frames.py "{VIDEO_DIR}" --out "{WORK}/input" '
    f'--num-frames {NUM_FRAMES} --max-size {MAX_IMAGE_SIZE}')

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
frames = sorted((WORK / "input").glob("*.jpg"))
preview = frames[:: max(1, len(frames) // 8)][:8]
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax in axes.flat:
    ax.axis("off")
for ax, f in zip(axes.flat, preview):
    ax.imshow(mpimg.imread(f))
    ax.set_title(f.name)
plt.tight_layout()
plt.show()

## ⑤ COLMAP: find the camera positions
COLMAP matches features across the frames and works out where the camera was for each one. This is the step most likely to fail when the video is poor, so **read the summary at the end**:
- **80–100% of frames placed:** 👍 continue.
- **under 70% of frames placed:** expect holes or blur. See *Troubleshooting* in the README and think about re-filming.

The result is saved to Drive as `colmap_data.zip`. If Colab disconnects later, this cell reloads it instead of running COLMAP again. Delete that zip if you change your video.

In [ ]:
#@title ⑤ Run COLMAP
colmap_zip = OUT_DIR / "colmap_data.zip"
if (WORK / "sparse/0/images.bin").exists():
    print("COLMAP already ran in this session. Skipping.")
elif colmap_zip.exists():
    print(f"Reusing earlier COLMAP result from {colmap_zip} (delete it to redo COLMAP).")
    run(f'mkdir -p "{WORK}" && unzip -q -o "{colmap_zip}" -d "{WORK}"')
else:
    run(f'python {REPO}/scripts/run_colmap.py --work "{WORK}" --matcher {MATCHER}')
    run(f'cd "{WORK}" && zip -q -r /content/colmap_data.zip images sparse && cp /content/colmap_data.zip "{colmap_zip}"')
    print(f"Saved {colmap_zip}")

## ⑥ Train the Gaussian splat
This step uses [gsplat](https://github.com/nerfstudio-project/gsplat).
- **The first run spends several minutes compiling GPU code** before the progress bar appears. This is normal.
- `loss` should fall over time. At 7,000 steps and at the end, the trainer prints quality scores: **PSNR** (higher is better; 25+ is good) and **SSIM** (closer to 1 is better).
- Keep this browser tab open while it trains. Free Colab may disconnect idle sessions.

In [ ]:
#@title ⑥ Train (30–60 min for 30000 steps on a T4)
RESULTS = WORK / "results_3dgs"
!cd {GSPLAT}/examples && python simple_trainer.py default \
    --data_dir "{WORK}" --data_factor 1 --result_dir "{RESULTS}" \
    --steps_scaler {STEPS_SCALER} --disable_viewer

## ⑦ Save the splat to Google Drive
This writes `outputs/<project>_splat.ply` (the splat) and a fly-around video, and copies the training checkpoints to Drive.

In [ ]:
#@title ⑦ Export
RESULTS = WORK / "results_3dgs"
splat_ply = OUT_DIR / f"{PROJECT_NAME}_splat.ply"
run(f'python {REPO}/scripts/export_ply.py "{RESULTS}" --out "{splat_ply}"')
run(f'mkdir -p "{OUT_DIR}/training_3dgs" && cp -r "{RESULTS}"/ckpts "{RESULTS}"/stats "{RESULTS}"/videos "{RESULTS}"/cfg.yml "{OUT_DIR}/training_3dgs/"')

from IPython.display import Video, display
videos = sorted((RESULTS / "videos").glob("traj_*.mp4"))
if videos:
    display(Video(str(videos[-1]), embed=True, width=640))
print(f"\nDone! Download {splat_ply.name} from Google Drive and open it in https://superspl.at/editor")

## Clean up your splat in SuperSplat
1. Download `<project>_splat.ply` from `My Drive/anatomy_splat/<project>/outputs/` to your computer.
2. Open [superspl.at/editor](https://superspl.at/editor) and drag the `.ply` onto the page.
3. To orbit, use left-drag. To pan, use right-drag. To zoom, scroll.
4. Use the **Sphere** or **Box** selection tool to select the table, background and floating "fog". Press **Delete**. The *Invert selection* option helps here: select the specimen, invert, then delete everything else.
5. Use **File → Export → PLY** to save the cleaned splat. Use **File → Publish** to get a shareable web link.

---
# ⑧ Optional: make a surface mesh
Gaussian splats look great but are not solid surfaces. To get a mesh (for 3D printing, Blender, 3D Slicer or MeshLab), this section:
1. trains a **2D Gaussian splat** (2DGS), a variant whose splats are flat discs that lie on the surface,
2. exports it so you can **delete the background in SuperSplat**,
3. turns your cleaned splat into a mesh with Poisson surface reconstruction.

It needs the COLMAP result from step ⑤. If Colab restarted, run cells ①–③ and ⑤ again first; step ⑤ reloads the saved result.

In [ ]:
#@title ⑧a Train a 2DGS splat (45–75 min)
RESULTS_2DGS = WORK / "results_2dgs"
!cd {GSPLAT}/examples && python simple_trainer_2dgs.py \
    --data-dir "{WORK}" --data-factor 1 --result-dir "{RESULTS_2DGS}" --model-type 2dgs \
    --depth-loss --depth-lambda 0.01 --normal-loss --normal-lambda 0.05 --dist-loss --dist-lambda 0.1 \
    --steps-scaler {STEPS_SCALER} --disable-viewer

In [ ]:
#@title ⑧b Export the 2DGS splat
RESULTS_2DGS = WORK / "results_2dgs"
run(f'python {REPO}/scripts/export_ply.py "{RESULTS_2DGS}" --out "{OUT_DIR}/{PROJECT_NAME}_2dgs.ply"')
print(f"""
Next:
  1. Download {PROJECT_NAME}_2dgs.ply from Drive and open it in https://superspl.at/editor
  2. Delete EVERYTHING except the specimen (table, background, floaters).
  3. Use File → Export → PLY (not "Compressed PLY") and name it {PROJECT_NAME}_2dgs_clean.ply
  4. Upload it to My Drive/anatomy_splat/{PROJECT_NAME}/outputs/ and run the next cell.
""")

In [ ]:
#@title ⑧c Build the mesh from your cleaned splat
#@markdown Higher depth gives more detail but a lumpier surface. Try 8–10.
POISSON_DEPTH = 9  #@param [8, 9, 10] {type:"raw"}
cleaned = OUT_DIR / f"{PROJECT_NAME}_2dgs_clean.ply"
if not cleaned.exists():
    raise FileNotFoundError(f"Upload your cleaned splat to {cleaned} first (see the previous cell).")
run("pip install -q open3d")
run(f'python {REPO}/scripts/make_mesh.py "{cleaned}" --out "{OUT_DIR}/{PROJECT_NAME}_mesh" --depth {POISSON_DEPTH}')
print("Open the .ply or .stl mesh in MeshLab, Blender, 3D Slicer or your slicer software.")